In [100]:
import sys
import math
from collections import defaultdict
from scipy.spatial import distance
import numpy as np
import itertools

In [ ]:
class Grid:
    def __init__(self, values):
        self.values = values
    
    def get_zeros_index(self):
        return [k for k,v in self.values.items() if v == 0]    
        
    def __int__(self):
        return int("".join(map(i for i in self.values.values())))

    def __str__(self):
        return str(self.values)

    def __hash__(self):
        return hash(str(self))

    def get(self,i):
        return self.values.get(i)

    def __eq__(self,other):
        return self.values == other.values

In [ ]:
class Grid:
    def __init__(self, values):
        self.values = values
    
    def get_zeros_index(self):
        return [k for k,v in self.values.items() if v == 0]    
        
    def __int__(self):
        return int("".join(map(str,(i for i in self.values.values()))))

    def __str__(self):
        return str(self.values)

    def __hash__(self):
        return hash(str(self))

    def get(self,i):
        return self.values.get(i)

    def __eq__(self,other):
        return self.values == other.values
    
    def copy(self):
        return Grid(self.values)

    def update(self,d):
        self.values.update(d)
        
    def items(self):
        return self.values.items()

In [ ]:
print(str(init))

In [117]:
a = {0: 1, 1: 0, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}
b = {0: 2, 1: 0, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}
c = {0: 3, 1: 0, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}
d = {0: 4, 1: 0, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}

g1 = Grid(a)
g2 = Grid(b)
g3 = Grid(c)
g4 = Grid(d)

In [ ]:
z = {"0":{"capture":{g1,g2},"non_capture":{g3,g4}}}



dict_values([{'capture': {<__main__.Grid object at 0x000001435E155650>, <__main__.Grid object at 0x000001436121C050>}, 'non_capture': {<__main__.Grid object at 0x000001435E157C10>, <__main__.Grid object at 0x0000014360DAE9D0>}}])

In [ ]:
import sys
import math
from collections import defaultdict
from scipy.spatial import distance
import numpy as np
import itertools

class Grid:
    def __init__(self, values):
        self.values = values
    
    def get_zeros_index(self):
        return [k for k,v in self.values.items() if v == 0]    
        
    def __int__(self):
        return int("".join(map(str,(i for i in self.values.values()))))

    def __str__(self):
        return str(self.values)

    def __hash__(self):
        return hash(str(self))

    def get(self,i):
        return self.values.get(i)

    def __eq__(self,other):
        return self.values == other.values
    
    def copy(self):
        return Grid(self.values)

    def update(self,d):
        self.values.update(d)
        
    def items(self):
        return self.values.items()


def get_patterns_list(height,width):
    cases = []
    for nr in range(height):
        for nc in range(width):
            cases.append((nr,nc))
            
    dist = distance.cdist(cases, cases, 'euclidean')
            
    groups = defaultdict(list)
    for key, value in np.argwhere(dist==1):
        groups[key].extend([key, value])

    groups = list(map(set,list(groups.values())))

    patterns = {}
    for k,g in enumerate(groups):
        combinations = []
        for i in range(3,6):
            c = list(itertools.combinations(g,i))
            combinations.append(c)
        combinations = list(itertools.chain(*combinations))
        combinations = [list(map(int,i)) for i in combinations if k in i]
        combinations = [[i for i in x if i != k] for x in combinations]
        patterns[k] = combinations

    patterns_list = []
    for k in patterns.keys():
        for v in patterns[k]:
            patterns_list.append((k,v))
        
    return patterns_list

def debug(txt):
    print(txt,file=sys.stderr,flush=True)

def process_values(values,grid,zero_index,pattern_index):
    s = sum(values)
    g = grid.copy()
    state = ""
    if (s <= 6) and (0 not in values):
        for t in pattern_index:
            g.update({t:0})
        g.update({zero_index:s})
        state = "capture"
    else:
        g.update({zero_index:1})
    return (state,g)

def compute(grid,patterns):
    '''
    La fonction "compute" traite chaque grille unitairement pour évaluer les prochains coups 
    Tout d'abord, elle identifie les zéros présent dans la grille. Pour chaque index où la valeur
    est égale à zéro, on récupère l'ensemble des patterns de capture associés (les patterns sont des tableaux d'entiers),
    '''
    states = {k:{"capture":{},"non_capture":{}} for k in range(9)}
    for zero,pattern_index in filter(lambda x : x[0] in grid.get_zeros_index(),patterns):
        g = grid.copy()
        s = sum(grid.get(p) for p in pattern_index)
        if s >= 6 and 0 not in pattern_index[-1]:
            g.update({p:0 for p in pattern_index})
            g.update({zero:1})
            states[zero]["capture"].add(g)
        else:
            states[zero]['non_capture'].add(g)
        
    
    
    zeros_indexes = grid.get_zeros_index()
    if len(zeros_indexes) != 0:
        result = []
        for zi in zeros_indexes:
            pattern_indexes = patterns.get(zi)                
            #Itérer sur les patterns de captures pour un zero_index donné et retourner les grilles calculées
            pv = [process_values(list(map(lambda x : grid.get(x),pi)),grid,zi,pi) for pi in pattern_indexes]
            if "capture" in list(map(lambda x : x[0],pv)):
                pv = [x for x in pv if x[0] == "capture"]
            result += set(map(lambda x : x[1],pv))
        return result
    else:
        return [grid]
    
def resolve(val,patterns):
    '''
    La fonction "resolve" est une fonction récursive pour appliquer le traitement sur les données.
    
    :param val: Le paramètre val est une liste de dictionnaire représentant les différentes grilles après chaque coup 
    :param patterns: La liste patterns contient l'ensemble des patterns de captures sur une grille de dimension donnée
    :return: La fonction retourne une liste de grille (dict)  
    '''
    #On récupère au sein de la fonction la variable "depth" pour monitorer le nombre de tour qu'il reste
    global depth
    final_results = []
    #Execution de la fonction "compute" dans un map pour évaluer toutes les options de jeux sur chaque grille dans la liste "val"
    print(list(map(lambda x : compute(x,patterns),val)))     
    final_results+=list(itertools.chain(*list(map(lambda x : compute(x,patterns),val))))
    if all(map(lambda x : 0 not in x.values.values(),final_results)):
        return final_results
    else:
        depth -= 1
        print(f"Tour - {depth}")
        if depth == 0:
            return final_results
        else:
            return resolve(final_results,patterns)

def concat_dict_values(x):
    return int("".join(map(str,list(x.values()))))

update = lambda j,k: (j+k)%2**30

  
   
patterns = [('0', [1, 3]),
 ('1', [0, 2]),
 ('1', [0, 4]),
 ('1', [2, 4]),
 ('1', [0, 2, 4]),
 ('2', [1, 5]),
 ('3', [0, 4]),
 ('3', [0, 6]),
 ('3', [4, 6]),
 ('3', [0, 4, 6]),
 ('4', [1, 3]),
 ('4', [1, 5]),
 ('4', [1, 7]),
 ('4', [3, 5]),
 ('4', [3, 7]),
 ('4', [5, 7]),
 ('4', [1, 3, 5]),
 ('4', [1, 3, 7]),
 ('4', [1, 5, 7]),
 ('4', [3, 5, 7]),
 ('4', [1, 3, 5, 7]),
 ('5', [8, 2]),
 ('5', [8, 4]),
 ('5', [2, 4]),
 ('5', [8, 2, 4]),
 ('6', [3, 7]),
 ('7', [8, 4]),
 ('7', [8, 6]),
 ('7', [4, 6]),
 ('7', [8, 4, 6]),
 ('8', [5, 7])]

init = {0: 3, 1: 0, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}
g = Grid(init)
depth = 24
r = list(map(lambda x : concat_dict_values(x),resolve([g],patterns)))
r = list(itertools.accumulate(r,update))[-1]

[303360010, 303360010, 303360010]
Tour - 23
[10061201, 10061201, 10061201, 10061201]
Tour - 22
[10061201, 10061201, 10061201, 10061201]
Tour - 21
[10061201, 10061201, 10061201, 10061201]
Tour - 20
[10061201, 10061201, 10061201, 10061201]
Tour - 19
[10061201, 10061201, 10061201, 10061201]
Tour - 18
[10061201, 10061201, 10061201, 10061201]
Tour - 17
[10061201, 10061201, 10061201, 10061201]
Tour - 16


KeyboardInterrupt: 

In [ ]:
final_results

In [ ]:
[{0: 3, 1: 0, 2: 3, 3: 3, 4: 6, 5: 0, 6: 1, 7: 0, 8: 2}, 
 {0: 3, 1: 1, 2: 0, 3: 3, 4: 6, 5: 2, 6: 0, 7: 3, 8: 0}, 
 {0: 0, 1: 4, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}, 
 {0: 3, 1: 0, 2: 1, 3: 3, 4: 6, 5: 2, 6: 0, 7: 3, 8: 0}, 
 {0: 3, 1: 1, 2: 0, 3: 3, 4: 6, 5: 2, 6: 0, 7: 3, 8: 0}, 
 {0: 3, 1: 0, 2: 1, 3: 3, 4: 6, 5: 2, 6: 0, 7: 3, 8: 0}, 
 {0: 3, 1: 0, 2: 0, 3: 0, 4: 6, 5: 2, 6: 6, 7: 0, 8: 0}, 
 {0: 3, 1: 0, 2: 0, 3: 3, 4: 6, 5: 0, 6: 0, 7: 0, 8: 5}]


In [ ]:
# init = {0: 0, 1: 6, 2: 0, 3: 2, 4: 2, 5: 2, 6: 1, 7: 6, 8: 1}
# depth = 20
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]

# if r == 322444322:
#     print("Test 1 : OK")
    
# init = {0: 5, 1: 0, 2: 6, 3: 4, 4: 5, 5: 0, 6: 0, 7: 6, 8: 4}
# depth = 20
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]
# if r == 951223336:
#     print("Test 2 : OK")
    
    
# init = {0: 5, 1: 5, 2: 5, 3: 0, 4: 0, 5: 5, 6: 5, 7: 5, 8: 5} 
# depth = 1
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]

# if r == 36379286:
#     print("Test 3 : OK")
    
    
# init = {0: 6, 1: 1, 2: 6, 3: 1, 4: 0, 5: 1, 6: 6, 7: 1, 8: 6}
# depth = 1
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]

# if r == 264239762:
#     print("Test 4 : OK")
    
    
# init = {0: 6, 1: 0, 2: 6, 3: 0, 4: 0, 5: 0, 6: 6, 7: 1, 8: 5}    
# depth = 8
# r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
# r = list(itertools.accumulate(r,update))[-1]
# if r == 76092874:
#     print("Test 5 : OK")
    
    
init = {0: 3, 1: 0, 2: 0, 3: 3, 4: 6, 5: 2, 6: 1, 7: 0, 8: 2}
depth = 24
r = list(map(lambda x : concat_dict_values(x),resolve([init],patterns)))
r = list(itertools.accumulate(r,update))[-1]
if r == 661168294:
    print("Test 6 : OK")
    

In [ ]:
300
362
102

In [ ]:
compute({0: 6, 1: 1, 2: 6, 3: 0, 4: 0, 5: 0, 6: 6, 7: 1, 8: 5},patterns)

In [ ]:
{0: 6, 1: 1, 2: 6, 3: 0, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5},

616
010
615

In [ ]:
616
000
615


606 616 616
020 100 001
615 615 615

In [ ]:
x = [{0: 6, 1: 1, 2: 6, 3: 1, 4: 0, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 1, 2: 6, 3: 0, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 2, 5: 0, 6: 6, 7: 0, 8: 5}, {0: 6, 1: 1, 2: 6, 3: 0, 4: 0, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 1, 2: 6, 3: 1, 4: 0, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 1, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 2, 5: 0, 6: 6, 7: 0, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 1, 4: 0, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 1, 2: 6, 3: 0, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 1, 4: 1, 5: 0, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 1, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 0, 5: 6, 6: 6, 7: 1, 8: 0}, {0: 6, 1: 1, 2: 6, 3: 0, 4: 0, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 1, 4: 0, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 1, 5: 1, 6: 6, 7: 1, 8: 5}, {0: 6, 1: 0, 2: 6, 3: 0, 4: 2, 5: 0, 6: 6, 7: 0, 8: 5}]
set(list(map(concat_dict_values,x)))

In [ ]:
set(list(map(concat_dict_values,z)))